# Does mlx-lm's loss count a padding token?

Notebook written by Claude at Fede's request (2 Nov 2026). Goal: check, with tests that don't depend on each other, whether mlx-lm's `default_loss` counts in the loss a token that doesn't exist (the first padding token), and what the consequences are. If confirmed, this notebook is the basis for reporting it.

Affected version: mlx-lm 0.32.0 (and `main` as of 2 Nov 2026).

## Why we suspect it, step by step

**1. Two numbers that don't add up.** For the bootstrap, Fede wrote `nll_ejemplo` in *Perplexity.ipynb*: it computes by hand the loss of each response, counting only the response tokens (the same thing `--mask-prompt` promises). On the 1,133 test examples, run1 gives **3.42**. `mlx_lm.lora --test` had given **3.237**. It's the same metric on the same data, so something counts differently.

**2. First hypothesis, discarded.** We thought mlx computed the prompt cut without `add_generation_prompt`, and so also counted the 3 tokens of `<|im_start|>assistant\n`. Reading `ChatDataset.process` we saw it doesn't: mlx uses `add_generation_prompt=True`, same as us. And when we tried it, the loss went up to 3.58 instead of down.

**3. Reading `default_loss`** (`mlx_lm/tuner/trainer.py`):
```python
inputs = batch[:, :-1]
targets = batch[:, 1:]
steps = mx.arange(1, targets.shape[1] + 1)
mask = mx.logical_and(steps >= lengths[:, 0:1], steps <= lengths[:, 1:])
```
`targets[j]` is token `j+1` of the sequence, and `steps[j] = j+1` is exactly that index. The mask keeps indices from `offset` to `L`, **both included**. But a sequence of `L` tokens has indices from `0` to `L-1`, so index `L` doesn't exist in the sequence: it's the first padding position. `iterate_batches` always pads with zeros and leaves at least one padding position (`1 + 32 * ceil(L/32) > L`). So every example adds one token too many to the loss: the **0**, which in Qwen's tokenizer is `"!"`.

Small example: a sequence of `L = 5` tokens (indices 0 to 4), response from `offset = 3`. The correct thing would be to count indices 3 and 4. The mask counts 3, 4 and **5**, and 5 is padding.

**4. The symptom in the model.** If this also happens during training, run1 has learned that after `<|im_end|>\n` comes `"!"`. With the first test example, run1 gives `"!"` at that position a probability of **99.4%**.

**5. Reproducing the number.** If we add that token 0 to `nll_ejemplo` and count it, the loss comes out at **3.2368**. mlx said 3.237.

**6. Precedent.** This same thing was reported in PR [ml-explore/mlx-lm#1697](https://github.com/ml-explore/mlx-lm/pull/1697) (August 2026), with the fix `steps < lengths` and two tests. It was closed without merging. The maintainer's only comment was "Fabricated issue by AI.". It was missing a real case.

## What's left to test, and what I expect before running

| Test | Question | If the bug is real, I expect |
|---|---|---|
| A | Does `default_loss` count one token too many per row? | `ntoks` = real tokens + 1 per row (4 extra in a batch of 4), and the last masked target is `"!"` |
| B | Is the `"!"` learned by training? | Base almost 0%; run1 goes up with the steps to ~99% |
| C | Fixing the mask, does mlx give the same as the by-hand calculation? | original `evaluate` ≈ 3.237; with `steps < lengths` ≈ by hand ≈ 3.42 |
| D | With the fixed mask, does training stop learning the `"!"`? | Same training, same seed: with the original it goes up; with the fixed one it stays as in base |

A and C test the code. B and D test the consequence in a real model.

## Setup

In [1]:
import json
from importlib.metadata import version

import numpy as np
import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim
from mlx_lm import load
from mlx_lm.tuner.datasets import CacheDataset, ChatDataset
from mlx_lm.tuner.trainer import default_loss, evaluate, iterate_batches
from mlx_lm.tuner.utils import linear_to_lora_layers

print("mlx-lm", version("mlx-lm"), "| mlx", version("mlx"))

MODELO = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
ADAPTADOR = "../adapters/run1"


def leer(ruta):
    with open(ruta) as f:
        return [json.loads(linea) for linea in f if linea.strip()]


test = leer("../data/dataset/session/test.jsonl")
train = leer("../data/dataset/session/train.jsonl")
print(len(test), "test examples,", len(train), "train")

mlx-lm 0.32.0 | mlx 0.32.4
1133 ejemplos de test, 5404 de train


In [2]:
def dataset_mlx(ejemplos, tokenizer):
    """The same dataset that mlx_lm.lora builds with --mask-prompt."""
    return CacheDataset(ChatDataset(ejemplos, tokenizer, chat_key="messages", mask_prompt=True))


def nll_ejemplo(model, tokenizer, messages):
    """Fede's function (Perplexity.ipynb): nll and number of tokens of the reply, without padding."""
    full = tokenizer.apply_chat_template(messages)
    prompt = tokenizer.apply_chat_template(messages[:-1], add_generation_prompt=True)
    logits = model(mx.array(full)[None]).astype(mx.float32)
    logprobs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    nll = -sum(logprobs[0, t - 1, full[t]].item() for t in range(len(prompt), len(full)))
    return nll, len(full) - len(prompt)


def prob_relleno(model, tokenizer, ejemplos):
    """Mean probability the model gives to token 0 ("!") right after the end of the reply."""
    probs = []
    for e in ejemplos:
        full = tokenizer.apply_chat_template(e["messages"])
        logits = model(mx.array(full)[None])[0, -1].astype(mx.float32)
        probs.append(mx.softmax(logits)[0].item())
    return float(np.mean(probs))


base, tokenizer = load(MODELO)
run1, _ = load(MODELO, adapter_path=ADAPTADOR)
print("Token 0 =", repr(tokenizer.decode([0])))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Token 0 = '!'


## Test A: how many tokens does `default_loss` count?

Why: it's the most direct test. I take a real batch just as mlx builds it (`iterate_batches` over the same dataset `mlx_lm.lora` uses), count the response tokens by hand (`L - offset` per row) and compare it with the `ntoks` that `default_loss` returns. Then I look at which exact tokens go into the loss in one row.

In [3]:
ds_test = dataset_mlx(test, tokenizer)
batch, lengths = next(iterate_batches(ds_test, batch_size=4, max_seq_length=2048))
offsets, longitudes = lengths[:, 0].tolist(), lengths[:, 1].tolist()

reales = sum(L - o for o, L in zip(offsets, longitudes))
_, ntoks = default_loss(run1, batch, lengths)
print("Real reply tokens:", reales)
print("Tokens counted by default_loss:", ntoks.item(), "-> extra", ntoks.item() - reales)

# Which targets go into the loss in the first row (same mask as default_loss)
fila = 0
objetivos = batch[fila, 1:].tolist()
steps = range(1, len(objetivos) + 1)
contados = [t for s, t in zip(steps, objetivos) if offsets[fila] <= s <= longitudes[fila]]
print("\nRow 0: real length", longitudes[fila], "| batch width with padding", batch.shape[1])
print("Last 3 targets it counts:", [tokenizer.decode([t]) for t in contados[-3:]])
print("Last 2 real tokens of the row:", [tokenizer.decode([t]) for t in batch[fila, longitudes[fila] - 2 : longitudes[fila]].tolist()])

Tokens de respuesta reales: 96
Tokens que cuenta default_loss: 100 -> sobran 4

Fila 0: longitud real 117 | ancho del lote con relleno 225
Últimos 3 objetivos que cuenta: ['<|im_end|>', '\n', '!']
Últimos 2 tokens reales de la fila: ['<|im_end|>', '\n']


**Result A:** confirmed. In a real batch of 4, the response has 96 tokens and `default_loss` counts 100: one extra per row. In row 0 (117 real tokens, batch padded to 225) the last three targets it counts are `<|im_end|>`, `\n` and `!`, while the last two real tokens are `<|im_end|>` and `\n`.

## Test B: is the `"!"` learned by training?

Why: run1's 99.4% could be a quirk of the base model and not something learned. The control is the base model without an adapter. Also, `adapters/run1` saves a checkpoint every 100 steps, so I can see whether the probability goes up as it trains. I use the first 50 test examples.

In [4]:
muestra = test[:50]
print(f"Base (no adapter): {prob_relleno(base, tokenizer, muestra):.4f}")
for paso in [100, 200, 500, 1000]:
    run1.load_weights(f"{ADAPTADOR}/{paso:07d}_adapters.safetensors", strict=False)
    print(f"run1 after {paso:>4} steps: {prob_relleno(run1, tokenizer, muestra):.4f}")

run1.load_weights(f"{ADAPTADOR}/adapters.safetensors", strict=False)  # back to the final adapter

Base (sin adaptador): 0.0000
run1 tras  100 pasos:  0.9932
run1 tras  200 pasos:  0.9951
run1 tras  500 pasos:  0.9984
run1 tras 1000 pasos:  0.9994


Model(
  (model): Qwen2Model(
    (embed_tokens): QuantizedEmbedding(151936, 896, group_size=64, bits=4, mode=affine)
    (layers.0): TransformerBlock(
      (self_attn): Attention(
        (q_proj): QuantizedLinear(input_dims=896, output_dims=896, bias=True, group_size=64, bits=4, mode=affine)
        (k_proj): QuantizedLinear(input_dims=896, output_dims=128, bias=True, group_size=64, bits=4, mode=affine)
        (v_proj): QuantizedLinear(input_dims=896, output_dims=128, bias=True, group_size=64, bits=4, mode=affine)
        (o_proj): QuantizedLinear(input_dims=896, output_dims=896, bias=False, group_size=64, bits=4, mode=affine)
        (rope): RoPE(64, traditional=False)
      )
      (mlp): MLP(
        (gate_proj): QuantizedLinear(input_dims=896, output_dims=4864, bias=False, group_size=64, bits=4, mode=affine)
        (down_proj): QuantizedLinear(input_dims=4864, output_dims=896, bias=False, group_size=64, bits=4, mode=affine)
        (up_proj): QuantizedLinear(input_dims=896, ou

**Result B:** confirmed. Probability of `!` right after the response, mean of the first 50 test examples: 0.0000 on the base model, 0.9932 after 100 steps of run1, 0.9951 after 200, 0.9984 after 500 and 0.9994 after 1,000. Training teaches it.

## Test C: fixing the mask, does mlx give the same as the by-hand calculation?

Why: if the only difference between mlx and `nll_ejemplo` is that token, changing `<=` to `<` (and nothing else) has to make mlx's `evaluate` give exactly the same as the by-hand calculation.

A detail to compare properly: `evaluate` drops the last incomplete batch (1,133 = 283 × 4 + 1), so it leaves one example out. I calculate by hand over the same 1,132, in the same order `iterate_batches` uses. It takes a few minutes.

In [5]:
def loss_arreglada(model, batch, lengths):
    """mlx-lm 0.32.0 default_loss with a single change: steps < lengths instead of steps <= lengths."""
    inputs = batch[:, :-1]
    targets = batch[:, 1:]
    logits = model(inputs)
    steps = mx.arange(1, targets.shape[1] + 1)
    mask = mx.logical_and(steps >= lengths[:, 0:1], steps < lengths[:, 1:])
    ce = nn.losses.cross_entropy(logits, targets) * mask
    ntoks = mask.sum()
    ce = ce.astype(mx.float32).sum() / ntoks
    return ce, ntoks


mlx_original = evaluate(run1, ds_test, batch_size=4, num_batches=-1)
mlx_arreglada = evaluate(run1, ds_test, batch_size=4, num_batches=-1, loss=loss_arreglada)

# The same examples evaluate uses: the same order as iterate_batches, without the last one
orden = sorted(range(len(ds_test)), key=ds_test.itemlen)
total_nll, total_tok = 0.0, 0
for i in orden[:-1]:
    nll, n = nll_ejemplo(run1, tokenizer, test[i]["messages"])
    total_nll += nll
    total_tok += n

print(f"mlx, original mask (<=):  {mlx_original:.4f}")
print(f"mlx, fixed mask (<):      {mlx_arreglada:.4f}")
print(f"by hand (nll_ejemplo):     {total_nll / total_tok:.4f}")

mlx, máscara original (<=): 3.2368
mlx, máscara arreglada (<): 3.4213
a mano (nll_ejemplo):       3.4213


**Result C:** confirmed. Over the same 1,132 test examples, mlx's `evaluate` gives 3.2368 with the original mask and 3.4213 with `steps < lengths`, exactly the by-hand `nll_ejemplo` (3.4213). The extra token is the whole difference.

## Test D: with the fixed mask, does it stop learning the `"!"`?

Why: A and C test the code, but what matters most to someone using the library is whether their model learns something weird. I train two identical LoRAs (same configuration as run1: rank 8, scale 20, 16 layers, Adam 1e-5, batches of 4, same data in the same order and same seed) for 200 steps. The only difference is the loss function. Then I measure the probability of `"!"` the same way as in B. It takes a few minutes.

In [6]:
def entrenar(loss_fn, pasos=200, semilla=0):
    mx.random.seed(semilla)
    modelo, _ = load(MODELO)
    modelo.freeze()
    linear_to_lora_layers(modelo, 16, {"rank": 8, "dropout": 0.0, "scale": 20.0})
    modelo.train()
    opt = optim.Adam(learning_rate=1e-5)
    perdida_y_grad = nn.value_and_grad(modelo, loss_fn)
    lotes = iterate_batches(dataset_mlx(train, tokenizer), batch_size=4, max_seq_length=2048, loop=True, seed=semilla)
    for _, (batch, lengths) in zip(range(pasos), lotes):
        (perdida, ntoks), grads = perdida_y_grad(modelo, batch, lengths)
        opt.update(modelo, grads)
        mx.eval(modelo.parameters(), opt.state)
    modelo.eval()
    return modelo


con_original = entrenar(default_loss)
con_arreglada = entrenar(loss_arreglada)
print(f"Base:                         {prob_relleno(base, tokenizer, muestra):.4f}")
print(f"200 steps, original mask:    {prob_relleno(con_original, tokenizer, muestra):.4f}")
print(f"200 steps, fixed mask:       {prob_relleno(con_arreglada, tokenizer, muestra):.4f}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Base:                         0.0000
200 pasos, máscara original:  0.9778
200 pasos, máscara arreglada: 0.0000


**Result D:** confirmed. Two identical 200-step trainings that differ only in the mask: probability of `!` 0.9778 with the original mask and 0.0000 with the fixed one (base 0.0000). With the fix the model doesn't learn the padding token.

## Conclusion

All four tests confirm it: mlx-lm 0.32.0's `default_loss` counts the first padding token after every response, the model learns to predict it, and the one-character fix (`<=` to `<`) removes both the extra token and its learning. Reported as [ml-explore/mlx-lm#2029](https://github.com/ml-explore/mlx-lm/issues/2029).

In this project every evaluation uses `nll_ejemplo`, so the numbers are not affected; every adapter was trained with mlx's loss as it is, so runs stay comparable (decision 15).

What it would mean if confirmed:
- **The metrics mlx-lm reports are biased.** Every example adds an extra token. In a model trained with mlx-lm that token is almost free, so its loss comes out lower than it really is. In the base model it's expensive, so it comes out higher. The fine-tuning improvement is exaggerated, and more so the shorter the responses are (here, 17.5 tokens on average).
- **The model learns to predict `"!"` after every response.** When generating it stops before, at `<|im_end|>`, so it doesn't show in the text, but the model spends capacity on it.
- **The fix is one character:** `steps <= lengths` → `steps < lengths`.

In [7]:
from mlx_lm import load
from mlx_lm.tuner.datasets import CacheDataset, ChatDataset
from mlx_lm.tuner.trainer import default_loss, iterate_batches

model, tokenizer = load("mlx-community/Qwen2.5-0.5B-Instruct-4bit")
data = [{"messages": [{"role": "user", "content": f"Say {w}"}, {"role": "assistant", "content": w}]}
        for w in ["hi", "hello there", "ok", "good morning"]]
ds = CacheDataset(ChatDataset(data, tokenizer, mask_prompt=True))
batch, lengths = next(iterate_batches(ds, batch_size=4, max_seq_length=2048))
_, ntoks = default_loss(model, batch, lengths)
print(int((lengths[:, 1] - lengths[:, 0]).sum()), ntoks.item())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

14 18
